# STEP 1 Evidence Harvester — Provenance-Hardened Final Version

Purpose:

> **Gather, preserve, organise, and export the evidence corpus for Step 1.**

This notebook does **not** adjudicate the science. It does not decide:
- construct validity;
- causal identification;
- mechanism;
- novelty;
- study architecture;
- neural modality;
- Step-1 closure.

Those decisions are made later from the exported corpus.

## Evidence hierarchy

The notebook preserves:

**Search plan**
→ **Search execution**
→ **Retrieval**
→ **Candidate**
→ **Screening decision**
→ **Verified publication**
→ **Experiment/study within publication**
→ **Atomic result**
→ **Evidence claim**
→ **Controlled proposition tag**

This keeps literature discovery separate from later scientific interpretation.

## 1. Install

In [ ]:
!pip -q install pandas numpy requests openpyxl biopython rapidfuzz

## 2. Mount Google Drive and create persistent folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import requests, re, json, os
from rapidfuzz import fuzz

ROOT = Path('/content/drive/MyDrive/Step1_Evidence_Corpus')
ROOT.mkdir(parents=True, exist_ok=True)

print("Persistent folder:", ROOT)

## 3. Controlled vocabularies

In [ ]:
ALLOWED_SEARCH_PURPOSE = {
    "DIRECT",
    "ADJACENT",
    "COUNTEREVIDENCE",
    "FOUNDATIONAL",
    "RECENT",
    "METHODOLOGICAL",
    "REPLICATION_BOUNDARY",
    "CODE_MATERIALS",
    "NOVELTY"
}

ALLOWED_SCREEN = {
    "INCLUDE",
    "EXCLUDE",
    "UNCLEAR",
    "DUPLICATE",
    "NOT_SCREENED"
}

ALLOWED_SUPPORT = {
    "supports",
    "counters",
    "qualifies",
    "background"
}

ALLOWED_DIRECTNESS = {
    "direct",
    "close analogue",
    "adjacent",
    "highly indirect"
}

ALLOWED_RESULT_STATUS = {
    "PRIMARY_PREREGISTERED",
    "SECONDARY_PREREGISTERED",
    "EXPLORATORY",
    "POST_HOC",
    "REPLICATION",
    "CONFIRMATORY_NOT_PREREGISTERED",
    "METHOD_OR_MANIPULATION_CHECK",
    "UNCLEAR",
    "NOT_APPLICABLE"
}

ALLOWED_DOMAIN_STATUS = {
    "NOT_STARTED",
    "SEARCHING",
    "SCREENING",
    "VERIFYING",
    "READY_FOR_AUDIT"
}

## 4. Schemas

In [ ]:
QUERY_PLAN_COLUMNS = [
    "query_plan_id","domain","search_purpose","planned_query",
    "preferred_sources","priority","rationale","status","notes"
]

SEARCH_COLUMNS = [
    "search_id","query_plan_id","domain","database","date_searched",
    "query","filters","search_purpose","direct_or_adjacent",
    "total_results_reported","results_retrieved","results_screened",
    "backward_checked","forward_checked","notes"
]

RETRIEVAL_COLUMNS = [
    "retrieval_id","search_id","candidate_id","retrieval_rank",
    "retrieval_method","database_record_id","notes"
]

CANDIDATE_COLUMNS = [
    "candidate_id","title","authors","year","doi","url",
    "screening_decision","exclusion_reason","duplicate_of",
    "linked_publication_id","notes"
]

PUBLICATION_COLUMNS = [
    "publication_id","citation","title","authors","year",
    "journal_or_venue","doi","authoritative_url","peer_reviewed",
    "primary_source_verified","full_text_checked","evidence_type",
    "preregistered","quality_flags","notes"
]

STUDY_COLUMNS = [
    "study_id","publication_id","study_label","sample_n_planned",
    "sample_n_analysed","design","population","manipulation",
    "comparator","timing_or_sessions","task",
    "primary_outcomes","secondary_outcomes",
    "study_analysis_status","attrition_or_exclusions","notes"
]

RESULT_COLUMNS = [
    "result_id","study_id","outcome","contrast_or_analysis",
    "analysis_n","effect_estimate","ci_or_se","test_statistic",
    "p_value","result_status","source_location",
    "near_verbatim_result","authors_interpretation","notes"
]

PROPOSITION_COLUMNS = [
    "proposition_tag","plain_language_question","domain",
    "active","notes"
]

EVIDENCE_CLAIM_COLUMNS = [
    "evidence_claim_id","result_id","domain","proposition_tag",
    "support_relation","what_it_establishes",
    "what_it_does_not_establish","directness",
    "boundary_condition","extractor_notes"
]

DOMAIN_COLUMNS = [
    "domain","search_status","notes"
]

## 5. Required Step-1 domains

In [ ]:
REQUIRED_DOMAINS = [
    "Bahg paradigm",
    "closed-loop recommender feedback",
    "human-AI recursive interaction",
    "real-world platform experiments",
    "normative learning under biased environments",
    "active information seeking",
    "compositional sampling data",
    "reciprocal sampling-learning loops",
    "category-learning systems",
    "learned attention and feature reweighting",
    "measurement reactivity",
    "temporal learning and accumulation",
    "persistence after algorithm removal",
    "sequential/adaptive experimental design",
    "control architectures",
    "static versus dynamic personalisation",
    "correction/reversal/recovery",
    "transfer",
    "computational models of behaviour",
    "model and parameter identifiability",
    "individual differences/susceptibility",
    "statistical/graph learning",
    "experience-dependent representational warping",
    "model-based cognitive neuroscience",
    "fMRI",
    "EEG/ERP",
    "eye tracking",
    "pupillometry",
    "fNIRS",
    "MEG",
    "multimodal neuroscience",
    "neural reliability and power",
    "pilot effects and winner's curse",
    "cross-study adaptive programmes",
    "online versus lab transportability",
    "programme feasibility",
    "novelty audit"
]

## 6. Pre-populated query matrix

This matrix prevents a domain from being represented by a single narrow query.

It is a **starting plan**, not a claim of completeness. Add, refine, and split queries as terminology emerges.

Each domain should ordinarily include a mix of:
- DIRECT;
- ADJACENT;
- COUNTEREVIDENCE;
- FOUNDATIONAL;
- RECENT;
- METHODOLOGICAL;
- REPLICATION/BOUNDARY CONDITION;
- CODE/MATERIALS where relevant.

In [ ]:
QUERY_PLAN_SEED = [
    # Bahg / source paradigm
    ("Bahg paradigm","DIRECT","Bahg personalized recommendation category learning information sampling","publisher;OSF;Google Scholar/Semantic Scholar if used","HIGH","Reconstruct source paradigm"),
    ("Bahg paradigm","CODE_MATERIALS","Bahg category learning recommender OSF code materials","OSF;publisher","HIGH","Verify generator/recommender implementation"),
    ("Bahg paradigm","REPLICATION_BOUNDARY","Bahg category learning replication recommendation sampling","Crossref;OpenAlex;publisher","HIGH","Locate replications/follow-ups"),

    # Recommender feedback loops
    ("closed-loop recommender feedback","DIRECT","recommender system feedback loops user behavior future recommendations","ACM;IEEE;OpenAlex","HIGH","Direct feedback-loop literature"),
    ("closed-loop recommender feedback","ADJACENT","preference pollution amplification recommender exposure diversity","ACM;IEEE;OpenAlex","HIGH","Mechanistic adjacent literature"),
    ("closed-loop recommender feedback","COUNTEREVIDENCE","recommender feedback loops no amplification diversity improvement experiment","ACM;IEEE;OpenAlex","MEDIUM","Actively seek boundary/null evidence"),
    ("closed-loop recommender feedback","RECENT","recommender feedback loop human experiment 2024 2025 2026","ACM;IEEE;OpenAlex","HIGH","Recent evidence"),

    # Human-AI recursive interaction
    ("human-AI recursive interaction","DIRECT","repeated human AI interaction bias amplification experiment","PubMed;OpenAlex;publisher","HIGH","Direct recursive human-AI evidence"),
    ("human-AI recursive interaction","COUNTEREVIDENCE","human AI repeated interaction debiasing correction no amplification","PubMed;OpenAlex","MEDIUM","Boundary/null evidence"),
    ("human-AI recursive interaction","RECENT","human AI feedback loop experiment 2024 2025 2026","PubMed;OpenAlex","HIGH","Recent work"),

    # Platforms
    ("real-world platform experiments","DIRECT","randomized algorithmic feed experiment recommendation platform users","OpenAlex;publisher","HIGH","Field RCTs"),
    ("real-world platform experiments","RECENT","Twitter X YouTube Spotify TikTok randomized recommendation experiment 2024 2025 2026","OpenAlex;publisher","HIGH","Current platform evidence"),
    ("real-world platform experiments","COUNTEREVIDENCE","algorithmic feed experiment null effect attitudes behavior","OpenAlex;publisher","MEDIUM","Null/boundary evidence"),

    # Normative learning
    ("normative learning under biased environments","DIRECT","category base rate learning unequal category frequency human experiment","PubMed;OpenAlex","HIGH","Base-rate adaptation"),
    ("normative learning under biased environments","ADJACENT","skewed training distribution generalization category learning","PubMed;OpenAlex","HIGH","Distribution shift"),
    ("normative learning under biased environments","FOUNDATIONAL","Bayesian category learning base rates environmental statistics","PubMed;OpenAlex","MEDIUM","Normative foundations"),
    ("normative learning under biased environments","COUNTEREVIDENCE","category learning base rate neglect biased sampling maladaptive","PubMed;OpenAlex","HIGH","Limits to rational adaptation"),
    ("normative learning under biased environments","METHODOLOGICAL","distribution shift covariate shift learning evaluation train test mismatch cognition","OpenAlex","MEDIUM","Evaluation framework"),

    # Active information seeking
    ("active information seeking","DIRECT","active information sampling category learning human","PubMed;OpenAlex","HIGH","Direct active category learning"),
    ("active information seeking","ADJACENT","expected information gain value of information sampling human learning","PubMed;OpenAlex","HIGH","Information-value literature"),
    ("active information seeking","ADJACENT","exploration exploitation information sampling belief learning","PubMed;OpenAlex","HIGH","Exploration-exploitation"),
    ("active information seeking","FOUNDATIONAL","optimal experiment design human hypothesis testing information search","PubMed;OpenAlex","MEDIUM","Foundational work"),
    ("active information seeking","COUNTEREVIDENCE","active sampling suboptimal confirmatory search biased belief","PubMed;OpenAlex","HIGH","Maladaptive sampling"),
    ("active information seeking","RECENT","active information sampling learning 2024 2025 2026","PubMed;OpenAlex","HIGH","Recent work"),

    # Compositional
    ("compositional sampling data","METHODOLOGICAL","compositional data psychology proportions simplex log ratio","PubMed;OpenAlex","HIGH","Methodology"),
    ("compositional sampling data","METHODOLOGICAL","ipsative measures reliability profile correlations psychology","PubMed;OpenAlex","HIGH","Reliability/interpretation"),

    # Reciprocal sampling-learning
    ("reciprocal sampling-learning loops","DIRECT","bidirectional information sampling concept learning Braunlich Love SEA","PubMed;OpenAlex","HIGH","Direct theory/evidence"),
    ("reciprocal sampling-learning loops","ADJACENT","belief dependent sampling self confirming beliefs learning","PubMed;OpenAlex","HIGH","Adjacent mechanism"),
    ("reciprocal sampling-learning loops","COUNTEREVIDENCE","exploration prevents sampling bias concept learning","PubMed;OpenAlex","MEDIUM","Boundary evidence"),

    # Category learning systems
    ("category-learning systems","FOUNDATIONAL","category learning rule based information integration exemplar prototype multiple systems review","PubMed;OpenAlex","HIGH","Core taxonomy"),
    ("category-learning systems","DIRECT","classification versus inference observational category learning experiment","PubMed;OpenAlex","HIGH","Task-format effects"),
    ("category-learning systems","DIRECT","feedback changes rule based information integration category learning","PubMed;OpenAlex","HIGH","Feedback effects"),
    ("category-learning systems","REPLICATION_BOUNDARY","category learning strategy switching memorization abstraction boundary conditions","PubMed;OpenAlex","HIGH","Strategy heterogeneity"),

    # Learned attention
    ("learned attention and feature reweighting","DIRECT","selective attention category learning diagnostic dimensions eye movements","PubMed;OpenAlex","HIGH","Direct learned attention"),
    ("learned attention and feature reweighting","DIRECT","learned inattention category learning extra dimensional shift","PubMed;OpenAlex","HIGH","Costs of narrowing"),
    ("learned attention and feature reweighting","RECENT","feature reweighting category learning memory attention 2024 2025 2026","PubMed;OpenAlex","HIGH","Recent work"),
    ("learned attention and feature reweighting","COUNTEREVIDENCE","selective attention improves category generalization efficient learning","PubMed;OpenAlex","HIGH","Adaptive narrowing"),

    # Measurement reactivity
    ("measurement reactivity","DIRECT","confidence rating reactivity category learning rule memorization","PubMed;OpenAlex","HIGH","Confidence reactivity"),
    ("measurement reactivity","DIRECT","judgment of learning reactivity category learning","PubMed;OpenAlex","HIGH","JOL reactivity"),
    ("measurement reactivity","FOUNDATIONAL","testing effect retrieval practice learning meta analysis","PubMed;OpenAlex","MEDIUM","Testing reactivity"),
    ("measurement reactivity","REPLICATION_BOUNDARY","confidence JOL reactivity boundary conditions 2024 2025 2026","PubMed;OpenAlex","HIGH","Boundary conditions"),

    # Temporal
    ("temporal learning and accumulation","DIRECT","multi session category learning longitudinal attention strategy","PubMed;OpenAlex","HIGH","Temporal trajectories"),
    ("temporal learning and accumulation","ADJACENT","category learning consolidation spacing saturation strategy switching","PubMed;OpenAlex","MEDIUM","Alternative trajectories"),
    ("temporal learning and accumulation","COUNTEREVIDENCE","nonmonotonic U shaped category learning repeated training","PubMed;OpenAlex","MEDIUM","Challenge monotonicity"),

    # Persistence
    ("persistence after algorithm removal","DIRECT","AI induced bias persistence after removal experiment","PubMed;OpenAlex","HIGH","Post-AI persistence"),
    ("persistence after algorithm removal","DIRECT","algorithm feed removal persistent behavior field experiment","OpenAlex;publisher","HIGH","Platform persistence"),
    ("persistence after algorithm removal","ADJACENT","delayed category learning generalization retention 24 hour","PubMed;OpenAlex","MEDIUM","Learning retention"),
    ("persistence after algorithm removal","COUNTEREVIDENCE","AI influence disappears after removal experiment","PubMed;OpenAlex","MEDIUM","Decay/null evidence"),

    # Adaptive design
    ("sequential/adaptive experimental design","METHODOLOGICAL","micro randomized trial proximal causal effects adaptive intervention","PubMed;OpenAlex","HIGH","Repeated randomisation"),
    ("sequential/adaptive experimental design","METHODOLOGICAL","SMART dynamic treatment regime sequential multiple assignment randomized trial","PubMed;OpenAlex","HIGH","Adaptive regimes"),
    ("sequential/adaptive experimental design","METHODOLOGICAL","time varying treatment causal inference state dependent intervention","PubMed;OpenAlex","HIGH","Causal estimands"),

    # Controls
    ("control architectures","DIRECT","active versus yoked category learning information sampling","PubMed;OpenAlex","HIGH","Yoked precedent"),
    ("control architectures","ADJACENT","yoked control replay noncontingent feedback agency learning","PubMed;OpenAlex","HIGH","Control limitations"),
    ("control architectures","COUNTEREVIDENCE","yoked exposure recipient state mismatch learning","PubMed;OpenAlex","MEDIUM","State-alignment concern"),
    ("control architectures","METHODOLOGICAL","matched exposure control active learning agency experimental design","PubMed;OpenAlex","MEDIUM","Design framework"),

    # Static vs dynamic personalization
    ("static versus dynamic personalisation","DIRECT","static versus dynamic personalization recommender human experiment","ACM;IEEE;OpenAlex","HIGH","Direct comparison"),
    ("static versus dynamic personalisation","DIRECT","frozen personalization adaptive recommendation user study","ACM;IEEE;OpenAlex","HIGH","Frozen policy"),
    ("static versus dynamic personalisation","RECENT","dynamic personalization longitudinal user experiment 2024 2025 2026","ACM;IEEE;OpenAlex","HIGH","Recent work"),

    # Correction
    ("correction/reversal/recovery","FOUNDATIONAL","reversal learning extinction relearning recovery category learning","PubMed;OpenAlex","HIGH","Foundational correction"),
    ("correction/reversal/recovery","DIRECT","learned inattention relevance shift category relearning","PubMed;OpenAlex","HIGH","Feature correction"),
    ("correction/reversal/recovery","ADJACENT","belief correction misinformation persistence correction","PubMed;OpenAlex","MEDIUM","Adjacent corrective information"),
    ("correction/reversal/recovery","COUNTEREVIDENCE","reversal learning spontaneous recovery renewal original memory persists","PubMed;OpenAlex","HIGH","Erasure alternatives"),

    # Transfer
    ("transfer","FOUNDATIONAL","Barnett Ceci transfer taxonomy near far transfer","PubMed;OpenAlex","HIGH","Transfer taxonomy"),
    ("transfer","DIRECT","category learning transfer learned attention novel structure","PubMed;OpenAlex","HIGH","Category transfer"),
    ("transfer","DIRECT","learned attention transfer feature weighting context","PubMed;OpenAlex","HIGH","Attention transfer"),
    ("transfer","COUNTEREVIDENCE","category learning transfer context specificity failure","PubMed;OpenAlex","MEDIUM","Boundary evidence"),

    # Computational models
    ("computational models of behaviour","FOUNDATIONAL","ALCOVE category learning attention model","PubMed;OpenAlex","MEDIUM","Foundational model"),
    ("computational models of behaviour","DIRECT","SEA information sampling concept learning model","PubMed;OpenAlex","HIGH","Sampling-learning model"),
    ("computational models of behaviour","DIRECT","AARM attention category learning eye tracking model","PubMed;OpenAlex","HIGH","Attention model"),
    ("computational models of behaviour","ADJACENT","Bayesian active learning category sampling model","PubMed;OpenAlex","MEDIUM","Alternative models"),

    # Identifiability
    ("model and parameter identifiability","METHODOLOGICAL","model recovery parameter recovery computational cognitive modeling","PubMed;OpenAlex","HIGH","Recovery methodology"),
    ("model and parameter identifiability","METHODOLOGICAL","identifiability latent cognitive parameters hierarchical models","PubMed;OpenAlex","HIGH","Parameter validity"),

    # Individual differences
    ("individual differences/susceptibility","DIRECT","individual differences category learning reliability","PubMed;OpenAlex","HIGH","Trait stability"),
    ("individual differences/susceptibility","DIRECT","susceptibility algorithmic influence individual differences experiment","PubMed;OpenAlex","HIGH","Algorithm susceptibility"),
    ("individual differences/susceptibility","METHODOLOGICAL","treatment effect heterogeneity reliability individual differences","PubMed;OpenAlex","HIGH","Moderation reliability"),
    ("individual differences/susceptibility","COUNTEREVIDENCE","stochastic path dependence apparent individual differences learning","PubMed;OpenAlex","MEDIUM","Alternative heterogeneity"),

    # Statistical/graph learning
    ("statistical/graph learning","DIRECT","statistical learning temporal community neural representation Schapiro","PubMed;OpenAlex","HIGH","Relational structure"),
    ("statistical/graph learning","DIRECT","graph learning network structure neural representation human fMRI","PubMed;OpenAlex","HIGH","Graph structure"),
    ("statistical/graph learning","RECENT","graph learning neural representation 2024 2025 2026","PubMed;OpenAlex","HIGH","Recent work"),

    # Representational warping
    ("experience-dependent representational warping","DIRECT","category learning perceptual warping neural representation","PubMed;OpenAlex","HIGH","Category-induced warping"),
    ("experience-dependent representational warping","ADJACENT","acquired distinctiveness equivalence neural similarity learning","PubMed;OpenAlex","MEDIUM","Similarity change"),
    ("experience-dependent representational warping","DIRECT","feature sharpening visual cortex category learning","PubMed;OpenAlex","HIGH","Feature coding"),

    # Model-based Cog Neuro
    ("model-based cognitive neuroscience","FOUNDATIONAL","model based cognitive neuroscience category learning model adjudication","PubMed;OpenAlex","HIGH","Methodological foundation"),
    ("model-based cognitive neuroscience","DIRECT","Mack Preston Love category learning fMRI exemplar prototype model","PubMed;OpenAlex","HIGH","Direct precedent"),
    ("model-based cognitive neuroscience","METHODOLOGICAL","RSA PCM encoding model representational model comparison","PubMed;OpenAlex","HIGH","Representational inference"),

    # fMRI
    ("fMRI","DIRECT","fMRI category learning multivariate representation RSA","PubMed;OpenAlex","HIGH","Direct modality precedent"),
    ("fMRI","METHODOLOGICAL","crossnobis crossvalidated Mahalanobis representational similarity reliability","PubMed;OpenAlex","HIGH","Distance estimator"),
    ("fMRI","COUNTEREVIDENCE","task fMRI reliability individual differences meta analysis","PubMed;OpenAlex","HIGH","Reliability limits"),

    # EEG
    ("EEG/ERP","DIRECT","EEG ERP category learning N250 P300 multivariate decoding","PubMed;OpenAlex","HIGH","Direct modality precedent"),
    ("EEG/ERP","DIRECT","time resolved RSA category representation EEG","PubMed;OpenAlex","HIGH","Temporal representation"),
    ("EEG/ERP","RECENT","EEG category learning strategy 2024 2025 2026","PubMed;OpenAlex","HIGH","Recent evidence"),

    # Eye tracking
    ("eye tracking","DIRECT","eye movements category learning diagnostic dimensions Rehder Hoffman","PubMed;OpenAlex","HIGH","Direct attention evidence"),
    ("eye tracking","DIRECT","gaze category learning feedback diagnosticity memory","PubMed;OpenAlex","HIGH","Learning dynamics"),
    ("eye tracking","METHODOLOGICAL","eye tracking reliability category learning attention","PubMed;OpenAlex","MEDIUM","Measurement quality"),

    # Pupillometry
    ("pupillometry","DIRECT","pupil category learning uncertainty prediction error","PubMed;OpenAlex","MEDIUM","Direct/near direct"),
    ("pupillometry","ADJACENT","pupil uncertainty learning rate surprise decision making","PubMed;OpenAlex","MEDIUM","Process evidence"),
    ("pupillometry","COUNTEREVIDENCE","pupillometry specificity luminance arousal confounds review","PubMed;OpenAlex","HIGH","Interpretive limitations"),

    # fNIRS
    ("fNIRS","DIRECT","fNIRS category learning rule similarity learning","PubMed;OpenAlex","HIGH","Direct modality evidence"),
    ("fNIRS","METHODOLOGICAL","fNIRS depth limitation reliability cognition review","PubMed;OpenAlex","MEDIUM","Capabilities/limits"),

    # MEG
    ("MEG","DIRECT","MEG category learning representation decoding","PubMed;OpenAlex","MEDIUM","Direct modality precedent"),
    ("MEG","METHODOLOGICAL","MEG time resolved category representation source localization","PubMed;OpenAlex","MEDIUM","Capabilities"),

    # Multimodal
    ("multimodal neuroscience","DIRECT","EEG fNIRS category learning multimodal","PubMed;OpenAlex","HIGH","Direct multimodal precedent"),
    ("multimodal neuroscience","ADJACENT","EEG eye tracking category learning attention","PubMed;OpenAlex","MEDIUM","Complementarity"),
    ("multimodal neuroscience","ADJACENT","fMRI eye tracking category learning representation","PubMed;OpenAlex","MEDIUM","Complementarity"),

    # Neural reliability/power
    ("neural reliability and power","METHODOLOGICAL","task fMRI reliability meta analysis ICC","PubMed;OpenAlex","HIGH","fMRI reliability"),
    ("neural reliability and power","METHODOLOGICAL","EEG ERP reliability decoding individual differences","PubMed;OpenAlex","HIGH","EEG reliability"),
    ("neural reliability and power","METHODOLOGICAL","multivariate neural representation reliability RSA","PubMed;OpenAlex","HIGH","Representational reliability"),
    ("neural reliability and power","METHODOLOGICAL","brain behavior correlation sample size winner curse neuroimaging","PubMed;OpenAlex","HIGH","Power/inflation"),

    # Pilot/winner curse
    ("pilot effects and winner's curse","METHODOLOGICAL","pilot study effect size inflation full trial winner curse","PubMed;OpenAlex","HIGH","Pilot inflation"),
    ("pilot effects and winner's curse","METHODOLOGICAL","selecting largest effect winner's curse psychology","PubMed;OpenAlex","HIGH","Selection bias"),

    # Cross-study adaptive programmes
    ("cross-study adaptive programmes","METHODOLOGICAL","discovery confirmation sequential research program preregistration","PubMed;OpenAlex","MEDIUM","Staged programme"),
    ("cross-study adaptive programmes","METHODOLOGICAL","internal pilot adaptive design effect size selection","PubMed;OpenAlex","MEDIUM","Cross-study decisions"),

    # Online-lab
    ("online versus lab transportability","DIRECT","online versus laboratory category learning replication","PubMed;OpenAlex","MEDIUM","Setting transport"),
    ("online versus lab transportability","ADJACENT","online cognitive experiment lab replication task context","PubMed;OpenAlex","MEDIUM","General transportability"),
    ("online versus lab transportability","METHODOLOGICAL","scanner task adaptation behavioral paradigm context effects","PubMed;OpenAlex","MEDIUM","Neural-setting adaptation"),

    # Feasibility
    ("programme feasibility","METHODOLOGICAL","longitudinal online experiment attrition repeated sessions psychology","PubMed;OpenAlex","MEDIUM","Participant burden"),
    ("programme feasibility","METHODOLOGICAL","MRI quality control exclusion attrition cognitive neuroscience","PubMed;OpenAlex","MEDIUM","Imaging burden"),
    ("programme feasibility","METHODOLOGICAL","EEG artifact rejection exclusion participant loss","PubMed;OpenAlex","MEDIUM","EEG burden"),

    # Novelty
    ("novelty audit","NOVELTY","behavior contingent recommendation category learning human experiment","OpenAlex;ACM;IEEE;PubMed","HIGH","Direct conjunction"),
    ("novelty audit","NOVELTY","adaptive recommender information sampling category learning persistence","OpenAlex;ACM;IEEE;PubMed","HIGH","Direct conjunction"),
    ("novelty audit","NOVELTY","algorithmic personalization neural category representation learning","OpenAlex;PubMed","HIGH","Neural conjunction")
]

query_plan_seed_df = pd.DataFrame(
    QUERY_PLAN_SEED,
    columns=[
        "domain","search_purpose","planned_query","preferred_sources",
        "priority","rationale"
    ]
)
query_plan_seed_df.insert(0, "query_plan_id", [f"QP{i:04d}" for i in range(1,len(query_plan_seed_df)+1)])
query_plan_seed_df["status"] = "PLANNED"
query_plan_seed_df["notes"] = ""

print("Seeded query plans:", len(query_plan_seed_df))
query_plan_seed_df.head()

## 7. Controlled proposition-tag registry

In [ ]:
PROPOSITION_SEED = [
    ("PT001","Does the Bahg paradigm demonstrate that personalised closed-loop exposure can alter category learning?","Bahg paradigm"),
    ("PT002","Does the current proposed task preserve the mechanism-relevant features of Bahg?","Bahg paradigm"),
    ("PT003","Can deviation on a balanced test be interpreted as cognitive distortion rather than adaptation to experienced statistics?","normative learning under biased environments"),
    ("PT004","Does sampling entropy index maladaptive information seeking?","active information seeking"),
    ("PT005","Are sampling amount, diversity, diagnosticity, information gain and attention distinct constructs?","active information seeking"),
    ("PT006","Are six-dimensional sampling proportions compositional/ipsative data requiring constrained-data methods?","compositional sampling data"),
    ("PT007","Do current beliefs or representations influence subsequent information sampling?","reciprocal sampling-learning loops"),
    ("PT008","Can selective sampling change the representation subsequently learned?","reciprocal sampling-learning loops"),
    ("PT009","Can confidence or JOL elicitation alter category-learning strategy?","measurement reactivity"),
    ("PT010","Can repeated full-information probes alter subsequent learning?","measurement reactivity"),
    ("PT011","Does repeated exposure necessarily imply monotonic accumulation?","temporal learning and accumulation"),
    ("PT012","Can algorithm/AI-induced effects persist after the system is removed?","persistence after algorithm removal"),
    ("PT013","Does baseline randomisation to a dynamic policy identify its total policy effect?","sequential/adaptive experimental design"),
    ("PT014","Does a Condition × Session analysis identify proximal causal effects of individual adaptive updates?","sequential/adaptive experimental design"),
    ("PT015","Does a frozen/static personalised policy isolate the incremental effect of ongoing self-contingent adaptation?","static versus dynamic personalisation"),
    ("PT016","Does yoked/replayed exposure cleanly isolate self-contingency?","control architectures"),
    ("PT017","Can agency or recipient-state alignment make yoked exposure psychologically non-equivalent?","control architectures"),
    ("PT018","Does Bahg involve online retraining of model parameters versus a fixed model responding to evolving user state/history?","Bahg paradigm"),
    ("PT019","Does behavioural convergence after correction establish recovery or erasure?","correction/reversal/recovery"),
    ("PT020","Is corrective learning following algorithmically generated exposure directly established in prior literature?","correction/reversal/recovery"),
    ("PT021","Does the proposed transfer paradigm constitute far/cross-domain transfer?","transfer"),
    ("PT022","Can latent feature weights or other behavioural parameters be reliably recovered from comparable data?","model and parameter identifiability"),
    ("PT023","Can behavioural latent parameters validly constrain later neural models?","model and parameter identifiability"),
    ("PT024","Can experience statistics reshape neural representational geometry?","experience-dependent representational warping"),
    ("PT025","Is FPN connectivity specifically privileged as the neural target for this phenomenon?","fMRI"),
    ("PT026","Is fMRI uniquely appropriate for the neural question?","fMRI"),
    ("PT027","Can EEG distinguish temporally separable feature/category learning processes relevant to this paradigm?","EEG/ERP"),
    ("PT028","Does eye tracking provide information about overt feature allocation beyond explicit information requests?","eye tracking"),
    ("PT029","Can fNIRS provide a theoretically useful cortical learning signal for relevant category-learning questions?","fNIRS"),
    ("PT030","Can neural data adjudicate cognitive models that fit behaviour similarly?","model-based cognitive neuroscience"),
    ("PT031","Does an A-versus-R neural group difference alone establish the cognitive mechanism?","model-based cognitive neuroscience"),
    ("PT032","Is stable individual susceptibility supported as a reliable trait in this problem space?","individual differences/susceptibility"),
    ("PT033","Can small pilot effects safely determine later effect-size assumptions?","pilot effects and winner's curse"),
    ("PT034","Can online behavioural effects be assumed to transport unchanged to scanner/EEG settings?","online versus lab transportability"),
    ("PT035","Are broad claims that algorithms influence behaviour, narrow some exposure, amplify bias, or persist after removal already established?","real-world platform experiments"),
    ("PT036","Is the full conjunction of self-contingent recommendation, known-ground-truth learning, information sampling, withdrawal and neural representation directly established?","novelty audit")
]

proposition_seed_df = pd.DataFrame(
    PROPOSITION_SEED,
    columns=["proposition_tag","plain_language_question","domain"]
)
proposition_seed_df["active"] = "yes"
proposition_seed_df["notes"] = ""

proposition_seed_df.head()

## 8. Persistent table creation/loading

In [ ]:
SCHEMAS = {
    "domains": DOMAIN_COLUMNS,
    "query_plan": QUERY_PLAN_COLUMNS,
    "propositions": PROPOSITION_COLUMNS,
    "searches": SEARCH_COLUMNS,
    "retrievals": RETRIEVAL_COLUMNS,
    "candidates": CANDIDATE_COLUMNS,
    "publications": PUBLICATION_COLUMNS,
    "studies": STUDY_COLUMNS,
    "results": RESULT_COLUMNS,
    "evidence_claims": EVIDENCE_CLAIM_COLUMNS
}

def csv_path(name):
    return ROOT / f"{name}.csv"

def load_or_create(name, columns):
    p = csv_path(name)
    if p.exists():
        df = pd.read_csv(p, dtype=str).fillna("")
        for c in columns:
            if c not in df.columns:
                df[c] = ""
        return df[columns]
    df = pd.DataFrame(columns=columns)
    df.to_csv(p, index=False)
    return df

tables = {name: load_or_create(name, cols) for name, cols in SCHEMAS.items()}

# Seed domains only if absent/empty
if len(tables["domains"]) == 0:
    tables["domains"] = pd.DataFrame({
        "domain": REQUIRED_DOMAINS,
        "search_status": "NOT_STARTED",
        "notes": ""
    })

# Seed query plan only if absent/empty
if len(tables["query_plan"]) == 0:
    tables["query_plan"] = query_plan_seed_df[QUERY_PLAN_COLUMNS].copy()

# Seed proposition registry only if absent/empty
if len(tables["propositions"]) == 0:
    tables["propositions"] = proposition_seed_df[PROPOSITION_COLUMNS].copy()

for name, df in tables.items():
    df.to_csv(csv_path(name), index=False)
    print(name, len(df))

## 9. ID generation and saving

In [ ]:
PREFIX = {
    "searches":"S",
    "retrievals":"RET",
    "candidates":"C",
    "publications":"P",
    "studies":"ST",
    "results":"R",
    "evidence_claims":"EC"
}

ID_COL = {
    "searches":"search_id",
    "retrievals":"retrieval_id",
    "candidates":"candidate_id",
    "publications":"publication_id",
    "studies":"study_id",
    "results":"result_id",
    "evidence_claims":"evidence_claim_id"
}

def save_table(name):
    tables[name].to_csv(csv_path(name), index=False)

def save_all():
    for name in tables:
        save_table(name)

def next_id(name):
    col = ID_COL[name]
    prefix = PREFIX[name]
    vals = tables[name][col].astype(str).tolist() if len(tables[name]) else []
    nums = []
    for v in vals:
        m = re.search(r"(\d+)$", v)
        if m:
            nums.append(int(m.group(1)))
    n = max(nums, default=0) + 1
    return f"{prefix}{n:04d}"

print({k:next_id(k) for k in PREFIX})

## 10. Normalization helpers

In [ ]:
def normalize_doi(x):
    x = str(x or "").strip().lower()
    x = re.sub(r"^https?://(dx\.)?doi\.org/", "", x)
    x = re.sub(r"^doi:\s*", "", x)
    return x.strip()

def normalize_title(x):
    return re.sub(r"[^a-z0-9]+"," ",str(x or "").lower()).strip()

## 11. Log an executed search

In [ ]:
def add_search(query_plan_id, database, date_searched, query=None, filters="",
               direct_or_adjacent="", total_results_reported="",
               results_retrieved="", results_screened="",
               backward_checked="", forward_checked="", notes=""):
    qp = tables["query_plan"][
        tables["query_plan"]["query_plan_id"].astype(str)==str(query_plan_id)
    ]
    if not len(qp):
        raise ValueError("Unknown query_plan_id")
    q = qp.iloc[0]
    sid = next_id("searches")
    row = {
        "search_id":sid,
        "query_plan_id":query_plan_id,
        "domain":q["domain"],
        "database":database,
        "date_searched":date_searched,
        "query":query if query is not None else q["planned_query"],
        "filters":filters,
        "search_purpose":q["search_purpose"],
        "direct_or_adjacent":direct_or_adjacent,
        "total_results_reported":total_results_reported,
        "results_retrieved":results_retrieved,
        "results_screened":results_screened,
        "backward_checked":backward_checked,
        "forward_checked":forward_checked,
        "notes":notes
    }
    tables["searches"] = pd.concat([tables["searches"],pd.DataFrame([row])],ignore_index=True)
    tables["query_plan"].loc[
        tables["query_plan"]["query_plan_id"].astype(str)==str(query_plan_id),
        "status"
    ] = "RUN"
    save_table("searches")
    save_table("query_plan")
    return sid

## 12. Add a candidate and retrieval link

In [ ]:
def add_candidate(title, authors="", year="", doi="", url="",
                  screening_decision="NOT_SCREENED",
                  exclusion_reason="", duplicate_of="",
                  linked_publication_id="", notes=""):
    if screening_decision not in ALLOWED_SCREEN:
        raise ValueError(f"screening_decision must be one of {ALLOWED_SCREEN}")
    cid = next_id("candidates")
    row = {
        "candidate_id":cid,
        "title":title,
        "authors":authors,
        "year":year,
        "doi":normalize_doi(doi),
        "url":url,
        "screening_decision":screening_decision,
        "exclusion_reason":exclusion_reason,
        "duplicate_of":duplicate_of,
        "linked_publication_id":linked_publication_id,
        "notes":notes
    }
    tables["candidates"] = pd.concat([tables["candidates"],pd.DataFrame([row])],ignore_index=True)
    save_table("candidates")
    return cid

def add_retrieval(search_id, candidate_id, retrieval_rank="",
                  retrieval_method="database search",
                  database_record_id="", notes=""):
    if str(search_id) not in set(tables["searches"]["search_id"].astype(str)):
        raise ValueError("Unknown search_id")
    if str(candidate_id) not in set(tables["candidates"]["candidate_id"].astype(str)):
        raise ValueError("Unknown candidate_id")
    rid = next_id("retrievals")
    row = {
        "retrieval_id":rid,
        "search_id":search_id,
        "candidate_id":candidate_id,
        "retrieval_rank":retrieval_rank,
        "retrieval_method":retrieval_method,
        "database_record_id":database_record_id,
        "notes":notes
    }
    tables["retrievals"] = pd.concat([tables["retrievals"],pd.DataFrame([row])],ignore_index=True)
    save_table("retrievals")
    return rid

def add_candidate_from_search(search_id, title, authors="", year="", doi="", url="",
                              database_record_id="", retrieval_rank="",
                              retrieval_method="database search",
                              screening_decision="NOT_SCREENED", notes=""):
    cid = add_candidate(
        title=title, authors=authors, year=year, doi=doi, url=url,
        screening_decision=screening_decision, notes=notes
    )
    add_retrieval(
        search_id=search_id, candidate_id=cid,
        retrieval_rank=retrieval_rank,
        retrieval_method=retrieval_method,
        database_record_id=database_record_id
    )
    return cid

## 13. Update screening decision

In [ ]:
def screen_candidate(candidate_id, decision, exclusion_reason="", duplicate_of=""):
    if decision not in ALLOWED_SCREEN:
        raise ValueError(f"decision must be one of {ALLOWED_SCREEN}")
    mask = tables["candidates"]["candidate_id"].astype(str)==str(candidate_id)
    if not mask.any():
        raise ValueError("Unknown candidate_id")
    if decision == "EXCLUDE" and not str(exclusion_reason).strip():
        raise ValueError("EXCLUDE requires an exclusion_reason")
    if decision == "DUPLICATE" and not str(duplicate_of).strip():
        raise ValueError("DUPLICATE requires duplicate_of")
    tables["candidates"].loc[mask,"screening_decision"] = decision
    tables["candidates"].loc[mask,"exclusion_reason"] = exclusion_reason
    tables["candidates"].loc[mask,"duplicate_of"] = duplicate_of
    save_table("candidates")

## 14. Add a verified publication and link it to one or more candidates

In [ ]:
def add_publication(citation, title, authors="", year="", journal_or_venue="",
                    doi="", authoritative_url="", peer_reviewed="",
                    primary_source_verified="", full_text_checked="",
                    evidence_type="", preregistered="", quality_flags="",
                    candidate_ids=None, notes=""):
    doi = normalize_doi(doi)

    # Reuse existing exact DOI
    if doi and len(tables["publications"]):
        existing = tables["publications"]["doi"].astype(str).map(normalize_doi)
        if (existing == doi).any():
            pid = tables["publications"].loc[existing==doi,"publication_id"].iloc[0]
        else:
            pid = None
    else:
        pid = None

    if pid is None:
        pid = next_id("publications")
        row = {
            "publication_id":pid,
            "citation":citation,
            "title":title,
            "authors":authors,
            "year":year,
            "journal_or_venue":journal_or_venue,
            "doi":doi,
            "authoritative_url":authoritative_url,
            "peer_reviewed":peer_reviewed,
            "primary_source_verified":primary_source_verified,
            "full_text_checked":full_text_checked,
            "evidence_type":evidence_type,
            "preregistered":preregistered,
            "quality_flags":quality_flags,
            "notes":notes
        }
        tables["publications"] = pd.concat(
            [tables["publications"],pd.DataFrame([row])],
            ignore_index=True
        )
        save_table("publications")

    if candidate_ids:
        if isinstance(candidate_ids, str):
            candidate_ids = [candidate_ids]
        for cid in candidate_ids:
            mask = tables["candidates"]["candidate_id"].astype(str)==str(cid)
            if not mask.any():
                raise ValueError(f"Unknown candidate_id: {cid}")
            tables["candidates"].loc[mask,"linked_publication_id"] = pid
            # Included if promoted to verified publication unless already explicitly excluded/duplicate
            current = tables["candidates"].loc[mask,"screening_decision"].iloc[0]
            if current in {"NOT_SCREENED","UNCLEAR"}:
                tables["candidates"].loc[mask,"screening_decision"] = "INCLUDE"
        save_table("candidates")

    return pid

## 15. Link an additional retrieval/candidate to an existing publication

In [ ]:
def link_candidate_to_publication(candidate_id, publication_id):
    if str(candidate_id) not in set(tables["candidates"]["candidate_id"].astype(str)):
        raise ValueError("Unknown candidate_id")
    if str(publication_id) not in set(tables["publications"]["publication_id"].astype(str)):
        raise ValueError("Unknown publication_id")
    mask = tables["candidates"]["candidate_id"].astype(str)==str(candidate_id)
    tables["candidates"].loc[mask,"linked_publication_id"] = publication_id
    if tables["candidates"].loc[mask,"screening_decision"].iloc[0] in {"NOT_SCREENED","UNCLEAR"}:
        tables["candidates"].loc[mask,"screening_decision"] = "INCLUDE"
    save_table("candidates")

## 16. Add an experiment/study within a publication

In [ ]:
def add_study(publication_id, study_label, sample_n_planned="",
              sample_n_analysed="", design="", population="",
              manipulation="", comparator="", timing_or_sessions="",
              task="", primary_outcomes="", secondary_outcomes="",
              study_analysis_status="", attrition_or_exclusions="", notes=""):
    if str(publication_id) not in set(tables["publications"]["publication_id"].astype(str)):
        raise ValueError("Unknown publication_id")
    stid = next_id("studies")
    row = {
        "study_id":stid,
        "publication_id":publication_id,
        "study_label":study_label,
        "sample_n_planned":sample_n_planned,
        "sample_n_analysed":sample_n_analysed,
        "design":design,
        "population":population,
        "manipulation":manipulation,
        "comparator":comparator,
        "timing_or_sessions":timing_or_sessions,
        "task":task,
        "primary_outcomes":primary_outcomes,
        "secondary_outcomes":secondary_outcomes,
        "study_analysis_status":study_analysis_status,
        "attrition_or_exclusions":attrition_or_exclusions,
        "notes":notes
    }
    tables["studies"] = pd.concat([tables["studies"],pd.DataFrame([row])],ignore_index=True)
    save_table("studies")
    return stid

## 17. Add an atomic result with result-level analysed N/status

In [ ]:
def add_result(study_id, outcome, contrast_or_analysis="",
               analysis_n="", effect_estimate="", ci_or_se="",
               test_statistic="", p_value="", result_status="UNCLEAR",
               source_location="", near_verbatim_result="",
               authors_interpretation="", notes=""):
    if str(study_id) not in set(tables["studies"]["study_id"].astype(str)):
        raise ValueError("Unknown study_id")
    if result_status not in ALLOWED_RESULT_STATUS:
        raise ValueError(f"result_status must be one of {ALLOWED_RESULT_STATUS}")
    rid = next_id("results")
    row = {
        "result_id":rid,
        "study_id":study_id,
        "outcome":outcome,
        "contrast_or_analysis":contrast_or_analysis,
        "analysis_n":analysis_n,
        "effect_estimate":effect_estimate,
        "ci_or_se":ci_or_se,
        "test_statistic":test_statistic,
        "p_value":p_value,
        "result_status":result_status,
        "source_location":source_location,
        "near_verbatim_result":near_verbatim_result,
        "authors_interpretation":authors_interpretation,
        "notes":notes
    }
    tables["results"] = pd.concat([tables["results"],pd.DataFrame([row])],ignore_index=True)
    save_table("results")
    return rid

## 18. Add a new proposition tag when needed

In [ ]:
def add_proposition_tag(question, domain, notes=""):
    # Generate next PT number
    vals = tables["propositions"]["proposition_tag"].astype(str).tolist()
    nums = []
    for v in vals:
        m = re.search(r"(\d+)$", v)
        if m:
            nums.append(int(m.group(1)))
    tag = f"PT{max(nums,default=0)+1:03d}"
    row = {
        "proposition_tag":tag,
        "plain_language_question":question,
        "domain":domain,
        "active":"yes",
        "notes":notes
    }
    tables["propositions"] = pd.concat(
        [tables["propositions"],pd.DataFrame([row])],
        ignore_index=True
    )
    save_table("propositions")
    return tag

## 19. Add an evidence claim using a controlled proposition tag

In [ ]:
def add_evidence_claim(result_id, domain, proposition_tag,
                       support_relation, what_it_establishes,
                       what_it_does_not_establish, directness,
                       boundary_condition="", extractor_notes=""):
    if str(result_id) not in set(tables["results"]["result_id"].astype(str)):
        raise ValueError("Unknown result_id")
    if str(proposition_tag) not in set(tables["propositions"]["proposition_tag"].astype(str)):
        raise ValueError("Unknown proposition_tag")
    if support_relation not in ALLOWED_SUPPORT:
        raise ValueError(f"support_relation must be one of {ALLOWED_SUPPORT}")
    if directness not in ALLOWED_DIRECTNESS:
        raise ValueError(f"directness must be one of {ALLOWED_DIRECTNESS}")

    ecid = next_id("evidence_claims")
    row = {
        "evidence_claim_id":ecid,
        "result_id":result_id,
        "domain":domain,
        "proposition_tag":proposition_tag,
        "support_relation":support_relation,
        "what_it_establishes":what_it_establishes,
        "what_it_does_not_establish":what_it_does_not_establish,
        "directness":directness,
        "boundary_condition":boundary_condition,
        "extractor_notes":extractor_notes
    }
    tables["evidence_claims"] = pd.concat(
        [tables["evidence_claims"],pd.DataFrame([row])],
        ignore_index=True
    )
    save_table("evidence_claims")
    return ecid

## 20. PubMed candidate discovery

In [ ]:
from Bio import Entrez
Entrez.email = "your_email@example.com"  # CHANGE THIS

def pubmed_search(query, retmax=100):
    h = Entrez.esearch(db="pubmed", term=query, retmax=retmax, sort="relevance")
    res = Entrez.read(h)
    ids = res["IdList"]
    total = int(res["Count"])
    if not ids:
        return total, pd.DataFrame()

    h = Entrez.esummary(db="pubmed", id=",".join(ids))
    rows = []
    for r in Entrez.read(h):
        rows.append({
            "pmid":str(r.get("Id","")),
            "title":str(r.get("Title","")),
            "journal":str(r.get("FullJournalName","")),
            "pubdate":str(r.get("PubDate","")),
            "article_ids":str(r.get("ArticleIds",""))
        })
    return total, pd.DataFrame(rows)

## 21. OpenAlex candidate discovery

In [ ]:
def openalex_search(query, per_page=100, mailto="your_email@example.com"):
    r = requests.get(
        "https://api.openalex.org/works",
        params={
            "search":query,
            "per-page":per_page,
            "mailto":mailto
        },
        timeout=30
    )
    r.raise_for_status()
    data = r.json()
    total = data.get("meta",{}).get("count","")
    rows = []
    for w in data.get("results",[]):
        rows.append({
            "openalex_id":w.get("id"),
            "title":w.get("display_name"),
            "year":w.get("publication_year"),
            "doi":normalize_doi(w.get("doi")),
            "type":w.get("type"),
            "cited_by_count":w.get("cited_by_count"),
            "landing_page":(w.get("primary_location") or {}).get("landing_page_url")
        })
    return total, pd.DataFrame(rows)

## 22. Candidate discovery is not full coverage

PubMed/OpenAlex are only discovery aids.

For relevant domains, manually search and log:
- ACM Digital Library;
- IEEE;
- publisher sites;
- OSF;
- supplementary materials/code;
- backward citations;
- forward citations;
- recommender/HCI proceedings.

The query plan tells you *what kinds of searches remain*. It does not claim that any one database is sufficient.

## 23. Duplicate-publication candidates

In [ ]:
def publication_duplicate_candidates(title_threshold=94):
    x = tables["publications"].reset_index(drop=True)
    rows = []
    for i in range(len(x)):
        for j in range(i+1,len(x)):
            doi_i = normalize_doi(x.loc[i,"doi"])
            doi_j = normalize_doi(x.loc[j,"doi"])
            same_doi = bool(doi_i and doi_j and doi_i==doi_j)
            ti = normalize_title(x.loc[i,"title"])
            tj = normalize_title(x.loc[j,"title"])
            sim = fuzz.ratio(ti,tj) if ti and tj else 0
            if same_doi or sim >= title_threshold:
                rows.append({
                    "publication_i":x.loc[i,"publication_id"],
                    "publication_j":x.loc[j,"publication_id"],
                    "same_doi":same_doi,
                    "title_similarity":sim,
                    "title_i":x.loc[i,"title"],
                    "title_j":x.loc[j,"title"]
                })
    return pd.DataFrame(rows)

## 24. Integrity checks for harvesting/provenance only

In [ ]:
def integrity_checks():
    issues = []

    # Duplicate IDs
    for name,col in [
        ("query_plan","query_plan_id"),
        ("searches","search_id"),
        ("retrievals","retrieval_id"),
        ("candidates","candidate_id"),
        ("publications","publication_id"),
        ("studies","study_id"),
        ("results","result_id"),
        ("evidence_claims","evidence_claim_id"),
        ("propositions","proposition_tag")
    ]:
        df = tables[name]
        if len(df):
            dup = df[df[col].astype(str).duplicated(keep=False)]
            for _,r in dup.iterrows():
                issues.append({"table":name,"id":r[col],"issue":"duplicate ID"})

    qp_ids = set(tables["query_plan"]["query_plan_id"].astype(str))
    search_ids = set(tables["searches"]["search_id"].astype(str))
    candidate_ids = set(tables["candidates"]["candidate_id"].astype(str))
    pub_ids = set(tables["publications"]["publication_id"].astype(str))
    study_ids = set(tables["studies"]["study_id"].astype(str))
    result_ids = set(tables["results"]["result_id"].astype(str))
    prop_ids = set(tables["propositions"]["proposition_tag"].astype(str))

    # Search -> query plan
    for _,r in tables["searches"].iterrows():
        if str(r["query_plan_id"]) not in qp_ids:
            issues.append({"table":"searches","id":r["search_id"],"issue":"missing query-plan link"})

    # Retrieval -> search + candidate
    for _,r in tables["retrievals"].iterrows():
        if str(r["search_id"]) not in search_ids:
            issues.append({"table":"retrievals","id":r["retrieval_id"],"issue":"missing search link"})
        if str(r["candidate_id"]) not in candidate_ids:
            issues.append({"table":"retrievals","id":r["retrieval_id"],"issue":"missing candidate link"})

    # Candidate screening / publication promotion
    for _,r in tables["candidates"].iterrows():
        cid = r["candidate_id"]
        decision = str(r["screening_decision"])
        if decision not in ALLOWED_SCREEN:
            issues.append({"table":"candidates","id":cid,"issue":"invalid screening decision"})
        if decision == "EXCLUDE" and not str(r["exclusion_reason"]).strip():
            issues.append({"table":"candidates","id":cid,"issue":"excluded candidate missing exclusion reason"})
        if decision == "DUPLICATE" and not str(r["duplicate_of"]).strip():
            issues.append({"table":"candidates","id":cid,"issue":"duplicate candidate missing duplicate_of"})
        if decision == "INCLUDE":
            pid = str(r["linked_publication_id"]).strip()
            if not pid:
                issues.append({"table":"candidates","id":cid,"issue":"included candidate not linked to verified publication"})
            elif pid not in pub_ids:
                issues.append({"table":"candidates","id":cid,"issue":"linked publication ID does not exist"})

    # Publication verification
    for _,r in tables["publications"].iterrows():
        pid = r["publication_id"]
        if not str(r["doi"]).strip() and not str(r["authoritative_url"]).strip():
            issues.append({"table":"publications","id":pid,"issue":"no DOI or authoritative URL"})

    # Publication -> study
    for _,r in tables["studies"].iterrows():
        if str(r["publication_id"]) not in pub_ids:
            issues.append({"table":"studies","id":r["study_id"],"issue":"missing publication link"})

    # Study -> result
    for _,r in tables["results"].iterrows():
        if str(r["study_id"]) not in study_ids:
            issues.append({"table":"results","id":r["result_id"],"issue":"missing study link"})
        if str(r["result_status"]) not in ALLOWED_RESULT_STATUS:
            issues.append({"table":"results","id":r["result_id"],"issue":"invalid result_status"})

    # Result -> evidence claim -> proposition
    for _,r in tables["evidence_claims"].iterrows():
        ecid = r["evidence_claim_id"]
        if str(r["result_id"]) not in result_ids:
            issues.append({"table":"evidence_claims","id":ecid,"issue":"missing result link"})
        if str(r["proposition_tag"]) not in prop_ids:
            issues.append({"table":"evidence_claims","id":ecid,"issue":"unknown proposition tag"})
        if str(r["support_relation"]) not in ALLOWED_SUPPORT:
            issues.append({"table":"evidence_claims","id":ecid,"issue":"invalid support relation"})
        if str(r["directness"]) not in ALLOWED_DIRECTNESS:
            issues.append({"table":"evidence_claims","id":ecid,"issue":"invalid directness"})
        if not str(r["what_it_establishes"]).strip():
            issues.append({"table":"evidence_claims","id":ecid,"issue":"missing what_it_establishes"})
        if not str(r["what_it_does_not_establish"]).strip():
            issues.append({"table":"evidence_claims","id":ecid,"issue":"missing what_it_does_not_establish"})

    return pd.DataFrame(issues)

## 25. Query-plan/domain progress summary

In [ ]:
def query_plan_summary():
    qp = tables["query_plan"].copy()
    searches = tables["searches"]
    rows = []
    for domain in REQUIRED_DOMAINS:
        qpd = qp[qp["domain"].astype(str).str.lower()==domain.lower()]
        sd = searches[searches["domain"].astype(str).str.lower()==domain.lower()]
        rows.append({
            "domain":domain,
            "planned_queries":len(qpd),
            "queries_run":int((qpd["status"].astype(str)=="RUN").sum()),
            "searches_logged":len(sd),
            "search_purposes_planned":"; ".join(sorted(set(qpd["search_purpose"].astype(str)))),
            "manual_domain_status":tables["domains"].loc[
                tables["domains"]["domain"].astype(str).str.lower()==domain.lower(),
                "search_status"
            ].iloc[0] if (
                tables["domains"]["domain"].astype(str).str.lower()==domain.lower()
            ).any() else ""
        })
    return pd.DataFrame(rows)

## 26. Proposition evidence bundle

In [ ]:
def proposition_bundle(proposition_tag):
    ecs = tables["evidence_claims"][
        tables["evidence_claims"]["proposition_tag"].astype(str)==str(proposition_tag)
    ].copy()
    if not len(ecs):
        return pd.DataFrame()

    r = ecs.merge(tables["results"],on="result_id",how="left")
    r = r.merge(tables["studies"],on="study_id",how="left",suffixes=("","_study"))
    r = r.merge(tables["publications"],on="publication_id",how="left",suffixes=("","_pub"))

    keep = [
        "proposition_tag","support_relation","directness",
        "publication_id","citation","title","doi","authoritative_url",
        "study_id","study_label","sample_n_analysed",
        "result_id","analysis_n","outcome","contrast_or_analysis",
        "effect_estimate","ci_or_se","test_statistic","p_value",
        "result_status","source_location","near_verbatim_result",
        "authors_interpretation",
        "what_it_establishes","what_it_does_not_establish",
        "boundary_condition"
    ]
    return r[[c for c in keep if c in r.columns]]

## 27. Publication provenance report

In [ ]:
def publication_provenance(publication_id):
    candidates = tables["candidates"][
        tables["candidates"]["linked_publication_id"].astype(str)==str(publication_id)
    ]
    if not len(candidates):
        return pd.DataFrame()

    x = candidates.merge(tables["retrievals"],on="candidate_id",how="left")
    x = x.merge(tables["searches"],on="search_id",how="left",suffixes=("","_search"))

    keep = [
        "publication_id","candidate_id","title","doi",
        "retrieval_id","retrieval_method","retrieval_rank",
        "search_id","query_plan_id","domain","database",
        "date_searched","query","search_purpose"
    ]
    x["publication_id"] = publication_id
    return x[[c for c in keep if c in x.columns]]

## 28. Export complete evidence corpus to Excel

In [ ]:
def export_excel(filename="Step1_Evidence_Corpus.xlsx"):
    path = ROOT / filename
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        tables["domains"].to_excel(w, sheet_name="Domains", index=False)
        tables["query_plan"].to_excel(w, sheet_name="Query_Plan", index=False)
        tables["propositions"].to_excel(w, sheet_name="Proposition_Tags", index=False)
        tables["searches"].to_excel(w, sheet_name="Searches", index=False)
        tables["retrievals"].to_excel(w, sheet_name="Retrievals", index=False)
        tables["candidates"].to_excel(w, sheet_name="Candidates", index=False)
        tables["publications"].to_excel(w, sheet_name="Publications", index=False)
        tables["studies"].to_excel(w, sheet_name="Experiments", index=False)
        tables["results"].to_excel(w, sheet_name="Results", index=False)
        tables["evidence_claims"].to_excel(w, sheet_name="Evidence_Claims", index=False)
        integrity_checks().to_excel(w, sheet_name="Integrity_Checks", index=False)
        query_plan_summary().to_excel(w, sheet_name="Query_Progress", index=False)
    print("Exported:", path)
    return path

## 29. What this notebook intentionally does not do

It does not:
- decide whether the literature supports a final proposition;
- decide construct validity;
- resolve contradictions;
- select a study design;
- rank EEG versus fMRI;
- infer mechanism;
- determine novelty;
- declare Step 1 closed.

The exported corpus is the input to the scientific audit.

The most important fields for later adjudication are:
- exact search provenance;
- screening decisions;
- verified publication;
- experiment-level sample/design;
- result-level analysed N;
- confirmatory/exploratory status;
- exact result/source location;
- what the result establishes;
- what it does not establish;
- boundary conditions;
- directness;
- controlled proposition tag.

## 30. Minimal synthetic integrity test

In [ ]:
def synthetic_smoke_test():
    # Test controlled vocabularies only; does not alter persistent tables.
    assert "DIRECT" in ALLOWED_SEARCH_PURPOSE
    assert "PRIMARY_PREREGISTERED" in ALLOWED_RESULT_STATUS
    assert "supports" in ALLOWED_SUPPORT
    assert "direct" in ALLOWED_DIRECTNESS

    # Query plan must cover every required domain
    covered = set(query_plan_seed_df["domain"].astype(str))
    missing = [d for d in REQUIRED_DOMAINS if d not in covered]
    assert not missing, f"Query-plan seed missing domains: {missing}"

    # Proposition tags unique
    assert proposition_seed_df["proposition_tag"].is_unique

    print("Synthetic smoke test passed.")

synthetic_smoke_test()